In [ ]:

######################
#stock_mastertableを作る
#初回のみ実行すること。通常実行しない
######################

import sqlite3
import pandas as pd
conn = sqlite3.connect("all_stocks.db")

#cleate tatbe　１回のみ
query = """
CREATE TABLE IF NOT EXISTS stock_master (
    ticker TEXT PRIMARY KEY,
    company_name TEXT,
    market TEXT
)
"""

conn.execute(query)
conn.commit()

In [ ]:
######################
#stock_priceを作る
#初回のみ実行すること。通常実行しない
######################

import sqlite3
import pandas as pd
conn = sqlite3.connect("all_stocks.db")

query="""
CREATE TABLE IF NOT EXISTS stock_price(
    ticker TEXT,
    date TEXT,
    open REAL,
    high REAL,
    low REAL,
    close REAL,
    volume INTEGER,
    PRIMARY KEY (ticker, date)
)
"""
conn.execute(query)
conn.commit()

In [ ]:
######################
#All tableを確認する
######################


pd.read_sql("""
SELECT *
FROM sqlite_master
WHERE type='table'
""", conn)

,type,name,tbl_name,rootpage,sql
0,table,stock_master,stock_master,2,"CREATE TABLE ""stock_master"" (\n""コード"" TEXT,\n ..."
1,table,stock_price,stock_price,189998,"CREATE TABLE ""stock_price"" (\n ticker TEXT,..."


In [ ]:
######################
#Table情報を確認する
######################

pd.read_sql("PRAGMA table_info(stock_master)", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,コード,TEXT,0,None,0
1,1,銘柄名,TEXT,0,None,0
2,2,市場・商品区分,TEXT,0,None,0


In [ ]:

##############################
#コードの銘柄や市場情報を取得する
##############################

import sqlite3
import pandas as pd
import xlrd

master_df = pd.read_excel("data\data_j.xls")
master_df = master_df[["コード", "銘柄名", "市場・商品区分"]]
master_df.head()

master_df.to_sql("stock_master",conn,if_exists="replace",index=False)

conn.commit()

market_list = ["プライム", "スタンダード", "グロース"]
market_df = master_df[
    master_df["市場・商品区分"].apply(lambda x: any(m in str(x) for m in market_list))]

tickers = market_df["コード"].tolist()
display(master_df.head(8))


,コード,銘柄名,市場・商品区分
0,1301,極洋,プライム（内国株式）
1,1305,ｉＦｒｅｅＥＴＦ ＴＯＰＩＸ（年１回決算型）,ETF・ETN
2,1306,ＮＥＸＴ ＦＵＮＤＳ ＴＯＰＩＸ連動型上場投信,ETF・ETN
3,1308,上場インデックスファンドＴＯＰＩＸ,ETF・ETN
4,1309,ＮＥＸＴ ＦＵＮＤＳ ＣｈｉｎａＡＭＣ・中国株式・上証５０連動型上場投信,ETF・ETN
5,130A,Ｖｅｒｉｔａｓ Ｉｎ Ｓｉｌｉｃｏ,グロース（内国株式）
6,1311,ＮＥＸＴ ＦＵＮＤＳ ＴＯＰＩＸ Ｃｏｒｅ ３０連動型上場投信,ETF・ETN
7,1319,ＮＥＸＴ ＦＵＮＤＳ 日経３００株価指数連動型上場投信,ETF・ETN


[1301, '130A', 1332, 1333, '135A']


prime_df = master_df[
    master_df["market"].str.contains("プライム", na=False)
]

prime_df.head()

In [ ]:
# =====================================
# 初期全件洗い替え専用（既存データ全削除）
# ※ 日次更新時は実行しない
# ====================================
import yfinance as yf
import pandas as pd
import xlrd

master_df = pd.read_excel("data\data_j.xls")
master_df = master_df[["コード", "銘柄名", "市場・商品区分"]]
master_df["コード"] = master_df["コード"].astype(str) + ".T"
master_df.head()

master_df.to_sql("stock_master",conn,if_exists="replace",index=False)
conn.commit()

market_list = ["プライム", "スタンダード", "グロース"]
market_df = master_df[
    master_df["市場・商品区分"].apply(lambda x: any(m in str(x) for m in market_list))]


tickers = market_df["コード"].tolist()
sample_tickers = tickers[:5000]

conn.execute("DELETE FROM stock_price")
conn.commit()

for i,ticker in enumerate(sample_tickers, start=1):
    try:
        df = yf.download(ticker, start="2018-01-01",progress=False)

        if df.empty:
            print(f"{ticker}: データなし")
            continue
    
        df.columns=df.columns.get_level_values(0)
        df.columns.name = None
        df["ticker"] = ticker
        df.reset_index(inplace=True)
        save_df = df[["ticker", "Date", "Open", "High", "Low", "Close", "Volume"]].copy()
        save_df.columns = [
            "ticker",
            "date",
            "open",
            "high",
            "low",
            "close",
            "volume"
        ]
        save_df["date"] = pd.to_datetime(save_df["date"]).dt.strftime("%Y-%m-%d")
    # print(save_df.head(2))

        save_df.to_sql(
            "stock_price",
            conn,
            if_exists="append",
            index=False
        )

        if i % 100 == 0:
            print(f"{i}件完了 / {len(sample_tickers)}件")


    except Exception as e:
        print(f"{ticker}: エラー {e}")

conn.commit()

check_df = pd.read_sql("SELECT * FROM stock_price LIMIT 10", conn)
display(check_df)

count_df = pd.read_sql(
    "SELECT COUNT(*) as cnt FROM stock_price",
    conn
)
display(count_df)




100件完了 / 3773件
200件完了 / 3773件
300件完了 / 3773件
400件完了 / 3773件
500件完了 / 3773件
600件完了 / 3773件
700件完了 / 3773件
800件完了 / 3773件
900件完了 / 3773件
1000件完了 / 3773件
1100件完了 / 3773件
1200件完了 / 3773件
1300件完了 / 3773件
1400件完了 / 3773件
1500件完了 / 3773件
1600件完了 / 3773件
1700件完了 / 3773件
1800件完了 / 3773件
1900件完了 / 3773件
2000件完了 / 3773件
2100件完了 / 3773件
2200件完了 / 3773件
2300件完了 / 3773件
2400件完了 / 3773件
2500件完了 / 3773件
2600件完了 / 3773件
2700件完了 / 3773件
2800件完了 / 3773件
2900件完了 / 3773件
3000件完了 / 3773件
3100件完了 / 3773件
3200件完了 / 3773件
3300件完了 / 3773件
3400件完了 / 3773件
3500件完了 / 3773件
3600件完了 / 3773件
3700件完了 / 3773件


,ticker,date,open,high,low,close,volume
0,1301.T,2018-01-01 00:00:00,3418.393555,3418.393555,3418.393555,3418.393555,0
1,1301.T,2018-01-02 00:00:00,3418.393555,3418.393555,3418.393555,3418.393555,0
2,1301.T,2018-01-03 00:00:00,3418.393555,3418.393555,3418.393555,3418.393555,0
3,1301.T,2018-01-04 00:00:00,3398.495981,3450.229526,3358.700946,3438.291016,61500
4,1301.T,2018-01-05 00:00:00,3446.250221,3470.127243,3410.434688,3454.209229,55300
5,1301.T,2018-01-08 00:00:00,3454.209229,3454.209229,3454.209229,3454.209229,0
6,1301.T,2018-01-09 00:00:00,3454.209229,3470.127243,3442.270717,3454.209229,26100
7,1301.T,2018-01-10 00:00:00,3454.208786,3549.716863,3454.208786,3525.839844,91300
8,1301.T,2018-01-11 00:00:00,3525.840023,3525.840023,3454.208962,3462.167969,48200
9,1301.T,2018-01-12 00:00:00,3458.188648,3458.188648,3394.516590,3398.496094,42500


,cnt
0,6934818


In [76]:
#日々の更新


import pandas as pd
import yfinance as yf

# マスタから全銘柄取得
tickers_df = pd.read_sql(
    "SELECT コード FROM stock_master",
    conn
)

tickers = tickers_df["コード"].tolist()

print("対象銘柄数:", len(tickers))

# =========================
# 全銘柄ループ
# =========================
for i, ticker in enumerate(tickers, start=1):

    try:
        print(f"{i}/{len(tickers)} 更新中: {ticker}")

        # =========================
        # 1. DB最新日取得
        # =========================
        query = f"""
        SELECT MAX(date) as max_date
        FROM stock_price
        WHERE ticker = '{ticker}'
        """

        last_date_df = pd.read_sql(query, conn)
        last_date = last_date_df.loc[0, "max_date"]

        # =========================
        # 2. 差分開始日決定
        # =========================
        if pd.isna(last_date):
            start_date = "2018-01-01"
        else:
            start_date = (
                pd.to_datetime(last_date)
                + pd.Timedelta(days=1)
            ).strftime("%Y-%m-%d")

        # =========================
        # 3. 株価取得
        # =========================
        df = yf.download(
            ticker,
            start=start_date,
            progress=False
        )

        if df.empty:
            print("  追加データなし")
            continue

        # =========================
        # 4. 整形
        # =========================
        df.columns = df.columns.get_level_values(0)
        df.columns.name = None

        df["ticker"] = ticker
        df.reset_index(inplace=True)

        save_df = df[
            ["ticker", "Date", "Open", "High", "Low", "Close", "Volume"]
        ].copy()

        save_df.columns = [
            "ticker",
            "date",
            "open",
            "high",
            "low",
            "close",
            "volume"
        ]

        # 日付正規化
        save_df["date"] = pd.to_datetime(
            save_df["date"]
        ).dt.strftime("%Y-%m-%d")

        # =========================
        # 5. DB追加
        # =========================
        records = list(
            save_df.itertuples(index=False, name=None)
        )

        insert_query = """
        INSERT OR IGNORE INTO stock_price
        (ticker, date, open, high, low, close, volume)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """

        conn.executemany(insert_query, records)

        # 100銘柄ごとに保存
        if i % 100 == 0:
            conn.commit()
            print(f"  {i}件コミット完了")

    except Exception as e:
        print(f"  エラー: {ticker} -> {e}")

# 最後に保存
conn.commit()

print("全銘柄更新完了")

対象銘柄数: 4439
1/4439 更新中: 1301.T
2/4439 更新中: 1305.T
3/4439 更新中: 1306.T
4/4439 更新中: 1308.T
5/4439 更新中: 1309.T
6/4439 更新中: 130A.T
7/4439 更新中: 1311.T


$1319.T: possibly delisted; no price data found  (1d 2026-04-11 -> 2026-04-12)

1 Failed download:
['1319.T']: possibly delisted; no price data found  (1d 2026-04-11 -> 2026-04-12)


8/4439 更新中: 1319.T


$131A.T: possibly delisted; no price data found  (1d 2026-04-11 -> 2026-04-12)

1 Failed download:
['131A.T']: possibly delisted; no price data found  (1d 2026-04-11 -> 2026-04-12)


  追加データなし
9/4439 更新中: 131A.T
  追加データなし
10/4439 更新中: 1320.T


KeyboardInterrupt: 

In [77]:
count_df=pd.read_sql("SELECT count (*) as cnt FROM stock_price",conn)

ticker_count = pd.read_sql(
    "SELECT COUNT(DISTINCT ticker) as cnt FROM stock_price",
    conn
)

query = """
SELECT ticker, COUNT(*) as cnt
FROM stock_price
GROUP BY ticker
ORDER BY cnt ASC
"""


display(count_df)
display(ticker_count)

check_df = pd.read_sql(query, conn)
display(check_df.head(20))
display(check_df.tail(20))

,cnt
0,7688681


,cnt
0,4438


,ticker,cnt
0,2069.T,12
1,494A.T,24
2,505A.T,30
3,510A.T,30
4,2068.T,31
5,512A.T,31
6,513A.T,31
7,504A.T,33
8,506A.T,34
9,507A.T,34


,ticker,cnt
4418,9969.T,2033
4419,9972.T,2033
4420,9973.T,2033
4421,9974.T,2033
4422,9976.T,2033
4423,9978.T,2033
4424,9979.T,2033
4425,9980.T,2033
4426,9982.T,2033
4427,9983.T,2033


In [78]:
import sqlite3
import pandas as pd

query = f"""
        SELECT * FROM stock_price
        WHERE ticker = "7203.T"
        order by date desc
        """

ex=pd.read_sql(query,conn)
display(ex.head(6))


,ticker,date,open,high,low,close,volume
0,7203.T,2026-04-10,3345.0,3363.0,3295.0,3319.0,16930900
1,7203.T,2026-04-09,3398.0,3399.0,3318.0,3331.0,20191400
2,7203.T,2026-04-08,3392.0,3418.0,3333.0,3384.0,27222600
3,7203.T,2026-04-07,3235.0,3264.0,3224.0,3252.0,14215500
4,7203.T,2026-04-06,3255.0,3297.0,3244.0,3247.0,11430400
5,7203.T,2026-04-03,3277.0,3316.0,3255.0,3255.0,11056900


In [79]:
dup_df = pd.read_sql("""
SELECT
    ticker,
    date(date) as norm_date,
    COUNT(*) as cnt
FROM stock_price
GROUP BY ticker, date(date)
HAVING COUNT(*) > 1
ORDER BY cnt DESC
LIMIT 20
""", conn)

display(dup_df)

,ticker,norm_date,cnt
